In [ ]:
"""
Script to build Quran embeddings for all verses and save them to
quran_embeddings.json. After running, your main app can load this file
directly without needing to recompute embeddings.

The embeddings are based on Arabic text and are language-agnostic.
The dictionary is read from the default language (German) to get Arabic keys.
"""

import os
import json
import time
from math import ceil
from openai import OpenAI

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------

# Source: German Quran dictionary (used to get Arabic verse keys)
QURAN_DICT_PATH = "../data/translations/quran/de.json"
# Output: Embeddings (language-agnostic, based on Arabic text)
QURAN_EMBEDDINGS_PATH = "../data/embeddings/quran_embeddings.json"
EMBEDDING_MODEL = "text-embedding-3-large"
BATCH_SIZE = 64

# -------------------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------------------

def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_json(path, data):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)

# -------------------------------------------------------------------
# OPENAI CLIENT
# -------------------------------------------------------------------

# Uses OPENAI_API_KEY environment variable or .env file
client = OpenAI()

# -------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------

quran_dict = load_json(QURAN_DICT_PATH)
if not quran_dict:
    raise RuntimeError(f"Quran dictionary empty or not found at {QURAN_DICT_PATH}")

print(f"Verses in quran_dict: {len(quran_dict)}")

quran_embeddings = load_json(QURAN_EMBEDDINGS_PATH)
print(f"Existing embeddings: {len(quran_embeddings)}")

# Stable ordering
all_verses = list(quran_dict.keys())

# Only verses that don't have embeddings yet
verses_to_embed = [v for v in all_verses if v not in quran_embeddings]
print(f"Missing verses (need embedding): {len(verses_to_embed)}")

if not verses_to_embed:
    print("All verses already have embeddings. Nothing to do. ✅")
    raise SystemExit(0)

In [ ]:
# -------------------------------------------------------------------
# EMBEDDING FUNCTION FOR A BATCH
# -------------------------------------------------------------------

def embed_batch(texts):
    """
    Takes a list of strings (verses),
    returns a list of embedding vectors (same order).
    """
    resp = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    # API guarantees resp.data is in the same order as input
    embeddings = [item.embedding for item in resp.data]
    return embeddings

# -------------------------------------------------------------------
# BATCH INDEXING
# -------------------------------------------------------------------

num_batches = ceil(len(verses_to_embed) / BATCH_SIZE)
print(f"Starting indexing in {num_batches} batches (batch size: {BATCH_SIZE})...")
print("You can stop this script anytime and restart later – ")
print("it will continue from where quran_embeddings.json left off.\n")

for batch_idx in range(num_batches):
    start = batch_idx * BATCH_SIZE
    end = start + BATCH_SIZE
    batch_verses = verses_to_embed[start:end]
    if not batch_verses:
        continue

    print(f"Batch {batch_idx + 1}/{num_batches} | Verses {start + 1}–{min(end, len(verses_to_embed))} "
          f"(total embeddings so far: {len(quran_embeddings)})")

    try:
        t0 = time.time()
        batch_embeddings = embed_batch(batch_verses)
        t1 = time.time()
    except Exception as e:
        print(f"\nError fetching embeddings for batch {batch_idx + 1}: {e}")
        print("Wait a moment, then restart the script – it will continue from here.\n")
        break

    # Add to dict
    for verse_text, emb in zip(batch_verses, batch_embeddings):
        quran_embeddings[verse_text] = emb

    # Save after each batch (checkpoint)
    save_json(QURAN_EMBEDDINGS_PATH, quran_embeddings)
    print(f"Batch {batch_idx + 1} saved. Duration: {t1 - t0:.2f}s\n")

print("Indexing complete (or stopped at error).")
print(f"Total embeddings saved: {len(quran_embeddings)}")
print("Done. ✅")

In [ ]:
# Verify the embeddings
print(f"Total embeddings: {len(quran_embeddings)}")
print(f"Sample verses: {list(quran_embeddings.keys())[:3]}")